# 7. Use and fine-tune the MACEField foundation model

[Open this notebook in Colab](https://colab.research.google.com/github/mdi-group/mace-field-tutorial/blob/main/tutorials/macefield/07_foundation_finetuning.ipynb)

This notebook downloads the official MACEField foundation checkpoint and its replay dataset from [release 1.0.2](https://github.com/mdi-group/mace-field/releases/tag/1.0.2). It first runs standalone inference on bundled BTO and SiO₂ structures, then shows an optional multi-head fine-tuning recipe on the bundled BaTiO₃ reference split.

The fine-tuning cell is opt-in because it needs a GPU and can take substantial time. It uses the released pt_head as the foundation head, creates an ft_head for the BTO labels, retains replay through pseudolabels, and inherits the foundation energy shifts. The small tutorial run is a workflow example, not a converged model.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PACKAGE_URL = "https://github.com/mdi-group/mace-field.git"
TUTORIAL_URL = "https://github.com/mdi-group/mace-field-tutorial.git"


def ensure_checkout(path, url):
    path = Path(path)
    if (path / ".git").is_dir():
        branch = subprocess.run(
            ["git", "-C", str(path), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch == "main":
            return
        subprocess.run(["git", "-C", str(path), "fetch", "origin", "main"], check=True)
        subprocess.run(
            ["git", "-C", str(path), "checkout", "-B", "main", "origin/main"],
            check=True,
        )
        return
    if path.exists() and any(path.iterdir()):
        raise RuntimeError(f"Refusing to clone over a non-empty directory: {path}")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", url, str(path)],
        check=True,
    )


if IN_COLAB:
    TUTORIAL_REPO = Path("/content/mace-field-tutorial")
    PACKAGE_REPO = Path("/content/mace-field")
    ensure_checkout(TUTORIAL_REPO, TUTORIAL_URL)
    ensure_checkout(PACKAGE_REPO, PACKAGE_URL)
else:
    CWD = Path.cwd().resolve()
    TUTORIAL_REPO = next(
        (
            path
            for path in (CWD, *CWD.parents)
            if (path / "tutorials/macefield/data").is_dir()
            and (path / "README.md").is_file()
        ),
        None,
    )
    if TUTORIAL_REPO is None:
        raise RuntimeError(
            "Run this notebook from a clone of mdi-group/mace-field-tutorial, "
            "or open it in Colab."
        )

    requested_repo = os.environ.get("MACEFIELD_REPO")
    if requested_repo:
        PACKAGE_REPO = Path(requested_repo).expanduser().resolve()
        if not (PACKAGE_REPO / "mace/modules/extensions.py").is_file():
            raise RuntimeError("MACEFIELD_REPO does not point to a MACE-Field checkout.")
        branch = subprocess.run(
            ["git", "-C", str(PACKAGE_REPO), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch != "main":
            raise RuntimeError("Set MACEFIELD_REPO to a checkout on the main branch.")
    else:
        candidates = [
            TUTORIAL_REPO.parent / "mace-field-main",
            TUTORIAL_REPO.parent / "mace-field",
        ]
        PACKAGE_REPO = None
        for candidate in candidates:
            if not (candidate / "mace/modules/extensions.py").is_file():
                continue
            branch = subprocess.run(
                ["git", "-C", str(candidate), "branch", "--show-current"],
                capture_output=True,
                text=True,
                check=False,
            ).stdout.strip()
            if branch == "main":
                PACKAGE_REPO = candidate.resolve()
                break
        if PACKAGE_REPO is None:
            PACKAGE_REPO = Path("/tmp/mace-field-main")
            ensure_checkout(PACKAGE_REPO, PACKAGE_URL)

os.environ["MACEFIELD_REPO"] = str(PACKAGE_REPO)
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
sys.path.insert(0, str(PACKAGE_REPO))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--editable", str(PACKAGE_REPO)],
    check=True,
)
os.chdir(TUTORIAL_REPO)

TUTORIAL_DIR = TUTORIAL_REPO / "tutorials/macefield"
DATA_DIR = TUTORIAL_DIR / "data"
print("Tutorial repository:", TUTORIAL_REPO)
print("MACE-Field source:", PACKAGE_REPO)


## Download and verify the released model and replay data

The release provides a MACEField model named MACEField-MH-0-omat-dielectric.model, built from a MACE multi-head MP-0 foundation model and fine-tuned on its OMAT head. Checksums pin the exact files used in this notebook.


In [ ]:
import hashlib
import urllib.request

RELEASE_BASE = "https://github.com/mdi-group/mace-field/releases/download/1.0.2"
ASSET_DIR = TUTORIAL_DIR / "runs" / "foundation_assets"
ASSET_DIR.mkdir(parents=True, exist_ok=True)

ASSETS = {
    "MACEField-MH-0-omat-dielectric.model": "f92e043aaf2cd8879919db8452503553fe7b608cb749d8d169dd96d4aa094aa2",
    "MP-dielectric-ferroelectric-replay-dataset.xyz": "4d7c3d3e8642b5691ccfb93bf3a41b7b89425d9018eeedef8cb73efabcfd350a",
}

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def download_verified(name, expected_sha256):
    path = ASSET_DIR / name
    if not path.is_file() or sha256_file(path) != expected_sha256:
        url = f"{RELEASE_BASE}/{name}"
        print("Downloading", name)
        urllib.request.urlretrieve(url, path)
    actual = sha256_file(path)
    if actual != expected_sha256:
        raise ValueError(f"Checksum mismatch for {name}: {actual}")
    print(name, "| SHA-256:", actual)
    return path

FOUNDATION_MODEL = download_verified(
    "MACEField-MH-0-omat-dielectric.model",
    ASSETS["MACEField-MH-0-omat-dielectric.model"],
)
REPLAY_DATA = download_verified(
    "MP-dielectric-ferroelectric-replay-dataset.xyz",
    ASSETS["MP-dielectric-ferroelectric-replay-dataset.xyz"],
)


## Run the released foundation head on BTO and SiO₂

These predictions use the checkpoint's pt_head. The structures and checkpoint are loaded here; no previous notebook output is needed.


In [ ]:
import numpy as np
import torch
from ase.io import read
from mace.calculators import MACECalculator

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BTO_SAMPLE = DATA_DIR / "macefield_batio3_reference_valid.extxyz"
SIO2_SAMPLE = DATA_DIR / "sio2/SiO2-preprocessed.extxyz"

def clean_structure(path):
    atoms = read(path, index=0)
    atoms.info.clear()
    for key in list(atoms.arrays):
        if key not in {"numbers", "positions"}:
            atoms.arrays.pop(key)
    return atoms

def foundation_predict(path, label):
    atoms = clean_structure(path)
    calc = MACECalculator(
        model_paths=str(FOUNDATION_MODEL),
        model_type="MACEField",
        head="pt_head",
        device=DEVICE,
        default_dtype="float32",
        electric_field=[0.0, 0.0, 0.0],
        compute_polarization=True,
        compute_becs=True,
        compute_polarizability=True,
    )
    atoms.calc = calc
    energy = atoms.get_potential_energy()
    print(label, "| atoms:", len(atoms), "| energy (eV):", energy)
    print("P (e/Å²):", np.asarray(calc.results["polarization"]))
    print("BEC shape:", np.asarray(calc.results["becs"]).reshape(len(atoms), 3, 3).shape)
    print("susceptibility:")
    print(np.asarray(calc.results["polarizability"]).reshape(3, 3))

foundation_predict(BTO_SAMPLE, "BaTiO3 reference frame")
foundation_predict(SIO2_SAMPLE, "alpha-quartz SiO2 frame")


## Optional: fine-tune a BTO head while replaying the foundation

The next cell selects replay structures, then writes a multi-head configuration with pt_head replay and a new ft_head trained on the 48-frame BTO training split. The 11-frame validation split stays separate. The selection step filters replay by chemical combinations, and the replay labels are generated from the foundation model.

Set MACEFIELD_RUN_FINETUNE=1 before running the cell to launch the one-epoch demonstration. Increase replay samples, epochs, and patience only after confirming the GPU memory budget and reviewing the split. The official release also includes an example fine-tuning script.


In [ ]:
import os
import subprocess
import sys
import time
import yaml

BTO_TRAIN = DATA_DIR / "macefield_batio3_reference_train.extxyz"
BTO_VALID = DATA_DIR / "macefield_batio3_reference_valid.extxyz"
RUN_DIR = TUTORIAL_DIR / "runs" / f"foundation_batio3_{time.strftime('%Y%m%d_%H%M%S')}"
RUN_DIR.mkdir(parents=True, exist_ok=False)
SELECTED_REPLAY = RUN_DIR / "selected_replay.xyz"
CONFIG = RUN_DIR / "finetune.yml"

NUM_REPLAY = int(os.environ.get("MACEFIELD_TUTORIAL_REPLAY_SAMPLES", "32"))
EPOCHS = int(os.environ.get("MACEFIELD_TUTORIAL_EPOCHS", "1"))
BATCH_SIZE = int(os.environ.get("MACEFIELD_TUTORIAL_BATCH_SIZE", "1"))

selector_command = [
    sys.executable, "-m", "mace.cli.fine_tuning_select",
    "--configs_pt", str(REPLAY_DATA),
    "--configs_ft", str(BTO_TRAIN),
    "--num_samples", str(NUM_REPLAY),
    "--subselect", "random",
    "--model", str(FOUNDATION_MODEL),
    "--model_type", "MACEField",
    "--output", str(SELECTED_REPLAY),
    "--head_pt", "pt_head",
    "--head_ft", "ft_head",
    "--filtering_type", "combinations",
    "--weight_pt", "1.0",
    "--weight_ft", "1.0",
    "--device", DEVICE,
    "--default_dtype", "float32",
    "--seed", "23",
]

finetune_config = {
    "name": "MACEField-foundation-BaTiO3-tutorial",
    "model": "MACEField",
    "loss": "universal_field",
    "foundation_model": str(FOUNDATION_MODEL),
    "foundation_head": "pt_head",
    "multiheads_finetuning": True,
    "pseudolabel_replay": True,
    "pt_train_file": str(SELECTED_REPLAY),
    "heads": {
        "ft_head": {
            "train_file": str(BTO_TRAIN),
            "valid_file": str(BTO_VALID),
        }
    },
    "valid_fraction": 0.2,
    "E0s": "foundation",
    "energy_weight": 1.0,
    "forces_weight": 10.0,
    "stress_weight": 1.0,
    "polarization_weight": 10.0,
    "becs_weight": 100.0,
    "polarizability_weight": 100.0,
    "compute_forces": True,
    "compute_stress": True,
    "compute_polarization": True,
    "compute_becs": True,
    "compute_polarizability": True,
    "batch_size": BATCH_SIZE,
    "valid_batch_size": 1,
    "num_workers": 0,
    "max_num_epochs": EPOCHS,
    "patience": max(3, EPOCHS),
    "eval_interval": 1,
    "default_dtype": "float32",
    "device": DEVICE,
    "seed": 23,
    "model_dir": str(RUN_DIR / "models"),
    "checkpoints_dir": str(RUN_DIR / "checkpoints"),
    "log_dir": str(RUN_DIR / "logs"),
    "results_dir": str(RUN_DIR / "results"),
    "work_dir": str(RUN_DIR / "work"),
}
for output_key in ("model_dir", "checkpoints_dir", "log_dir", "results_dir", "work_dir"):
    Path(finetune_config[output_key]).mkdir(parents=True, exist_ok=True)
CONFIG.write_text(yaml.safe_dump(finetune_config, sort_keys=False))
print("Fine-tuning configuration:", CONFIG)
print("Selected replay output:", SELECTED_REPLAY)
print("Set MACEFIELD_RUN_FINETUNE=1 to run with", NUM_REPLAY, "replay frames and", EPOCHS, "epoch(s).")

if os.environ.get("MACEFIELD_RUN_FINETUNE", "0") == "1":
    subprocess.run(selector_command, cwd=PACKAGE_REPO, check=True)
    subprocess.run(
        [sys.executable, "-m", "mace.cli.run_train", "--config", str(CONFIG)],
        cwd=PACKAGE_REPO,
        check=True,
    )
    print("Fine-tuned models:", list((RUN_DIR / "models").glob("*.model")))
else:
    print("Training skipped. Foundation inference above ran independently.")


## Interpretation

The released checkpoint and replay set are separate, pinned inputs to this notebook. Fine-tuning keeps the foundation's replay head while introducing a target head for the local BTO labels. The example's one epoch is only a runnable configuration demonstration; it is not a converged transfer-learning result.
